# Tutorial 5: Developing an Extension (Dev)

This tutorial shows how to create custom extensions for scivianna panels, adding new functionality and UI elements.

## Introduction

**Extensions** in scivianna are modular components that add new functionality to panels. They can:
- Add custom UI widgets (buttons, sliders, checkboxes)
- Add computation logic that runs when parameters change
- Modify panel rendering behavior
- Interact with the slave's data

## Extension Architecture

Extensions are built on two core classes:
- **`Extension`**: Base class for all extensions, providing UI widget management and computation hooks
- **`ExtensionWidget`**: Qt-based widget that displays extension parameters in a collapsible card

## Extension Lifecycle

1. **Initialization**: Parameters are created with `create_param`
2. **UI Display**: Widget renders parameters in a collapsible card
3. **User Interaction**: User changes parameters
4. **Computation Trigger**: `compute` method is called
5. **Result Processing**: `process_result` handles computation results
6. **Display Update**: `display_result` updates the panel visualization

## Step 1: Simple Extension with UI Widgets

A minimal extension that adds a checkbox and a line edit.

In [ ]:
from scivianna.extension.extension import Extension

class MySimpleExtension(Extension):
    """A simple extension with basic UI widgets."""
    
    def __init__(self, panel=None):
        # Call parent constructor
        super().__init__(panel)
        
        # Create parameters with default values
        self.create_param("show_labels", True, bool)
        self.create_param("opacity", 1.0, float)
        self.create_param("text_input", "Hello", str)
    
    def compute(self):
        """Called when any parameter changes."""
        show_labels = self.get_param("show_labels")
        opacity = self.get_param("opacity")
        text = self.get_param("text_input")
        
        print(f"Extension updated:")
        print(f"  Show labels: {show_labels}")
        print(f"  Opacity: {opacity}")
        print(f"  Text: {text}")
        
        # Return a result dict for process_result
        return {
            "show_labels": show_labels,
            "opacity": opacity,
            "text": text,
        }
    
    def process_result(self, result):
        """Process the computation result."""
        print(f"Processing result: {result}")
        # Store result for display
        self._last_result = result
    
    def display_result(self, result):
        """Update the panel with the result."""
        if self.panel:
            # Update panel properties based on extension parameters
            if hasattr(self.panel, 'set_opacity'):
                self.panel.set_opacity(result["opacity"])
            print(f"Display updated with: {result}")

# Usage:
# ext = MySimpleExtension(panel=my_panel)
# panel.add_extension(ext)

## Step 2: Different Parameter Types

Extensions support various parameter types with different UI widgets.

In [ ]:
from scivianna.extension.extension import Extension

class RichParameterExtension(Extension):
    """Extension demonstrating different parameter types."""
    
    def __init__(self, panel=None):
        super().__init__(panel)
        
        # String parameter -> LineEdit widget
        self.create_param("name", "MyExtension", str)
        
        # Boolean parameter -> CheckBox widget
        self.create_param("enabled", True, bool)
        
        # Float parameter -> DoubleSpinBox (decimal slider)
        self.create_param("threshold", 0.5, float, min=0.0, max=1.0, step=0.01)
        
        # Int parameter -> SpinBox (integer slider)
        self.create_param("iterations", 10, int, min=1, max=100, step=1)
        
        # String list parameter -> ComboBox
        self.create_param(
            "mode",
            "fast",  # default
            str,
            items=["fast", "balanced", "accurate"],  # dropdown options
        )
        
        # Color parameter -> QColorButton
        self.create_param("color", (255, 0, 0), tuple)  # RGB tuple
        
        # Grouped parameters -> Use name prefix with separator
        self.create_param("display.x_axis", True, bool)
        self.create_param("display.y_axis", True, bool)
        self.create_param("display.z_axis", False, bool)
    
    def compute(self):
        """Called when any parameter changes."""
        params = {
            "name": self.get_param("name"),
            "enabled": self.get_param("enabled"),
            "threshold": self.get_param("threshold"),
            "iterations": self.get_param("iterations"),
            "mode": self.get_param("mode"),
            "color": self.get_param("color"),
            "x_axis": self.get_param("display.x_axis"),
            "y_axis": self.get_param("display.y_axis"),
            "z_axis": self.get_param("display.z_axis"),
        }
        print(f"Parameters updated: {params}")
        return params

# Usage:
# ext = RichParameterExtension(panel=my_panel)
# panel.add_extension(ext)

## Step 3: Grouped Parameters

Use the `group` parameter to group related parameters under a collapsible section.

In [ ]:
from scivianna.extension.extension import Extension

class GroupedExtension(Extension):
    """Extension with grouped parameters."""
    
    def __init__(self, panel=None):
        super().__init__(panel)
        
        # Group 1: Display settings
        self.create_param("display.color", (255, 0, 0), tuple, group="Display")
        self.create_param("display.opacity", 1.0, float, min=0.0, max=1.0, group="Display")
        self.create_param("display.visible", True, bool, group="Display")
        
        # Group 2: Geometry settings
        self.create_param("geometry.scale", 1.0, float, min=0.1, max=10.0, group="Geometry")
        self.create_param("geometry.rotation", 0.0, float, min=0.0, max=360.0, group="Geometry")
        
        # Group 3: Advanced settings
        self.create_param("advanced.smoothing", False, bool, group="Advanced")
        self.create_param("advanced.antialiasing", True, bool, group="Advanced")
    
    def compute(self):
        """Called when any parameter changes."""
        result = {}
        for key in self.params:
            result[key] = self.get_param(key)
        print(f"Grouped parameters: {result}")
        return result

# The ExtensionWidget automatically groups parameters by the 'group' parameter.
# Each group appears as a collapsible section in the UI.

## Step 4: Accessing Slave Data

Extensions can access the slave's data through `self.panel.slave`.

In [ ]:
from scivianna.extension.extension import Extension

class DataAnalysisExtension(Extension):
    """Extension that analyzes slave data."""
    
    def __init__(self, panel=None):
        super().__init__(panel)
        
        self.create_param("field", "temperature", str)
        self.create_param("stat_type", "mean", str, items=["mean", "min", "max", "std"])
    
    def compute(self):
        """Analyze slave data and compute statistics."""
        # Access the slave through the panel
        if not self.panel or not hasattr(self.panel, 'slave'):
            return {"error": "No slave available"}
        
        slave = self.panel.slave
        
        # Get interface from slave
        interface = slave.get_interface()
        
        # Get available labels
        labels = interface.get_labels() if hasattr(interface, 'get_labels') else []
        
        # Get the selected field values
        field = self.get_param("field")
        stat_type = self.get_param("stat_type")
        
        # Example: compute statistics from interface data
        result = {
            "available_labels": labels,
            "field": field,
            "stat_type": stat_type,
            "computed": True,
        }
        
        # If the interface has data, compute actual stats
        if hasattr(interface, 'values') and field in interface.values:
            import numpy as np
            data = interface.values[field]
            if stat_type == "mean":
                result["value"] = float(np.mean(data))
            elif stat_type == "min":
                result["value"] = float(np.min(data))
            elif stat_type == "max":
                result["value"] = float(np.max(data))
            elif stat_type == "std":
                result["value"] = float(np.std(data))
        
        print(f"Analysis result: {result}")
        return result

# Usage:
# ext = DataAnalysisExtension(panel=my_panel)
# panel.add_extension(ext)

## Step 5: Complete Example - Color Map Editor

A practical extension that lets users customize the color map of a 2D panel.

In [ ]:
"""
Complete Color Map Editor Extension

This extension allows users to customize the color map, min/max values,
and reversal of the color mapping for a 2D panel.
"""

import numpy as np
from scivianna.extension.extension import Extension


class ColorMapEditor(Extension):
    """Extension for customizing color map settings."""
    
    # Available matplotlib colormaps
    COLORMAPS = [
        "viridis", "plasma", "inferno", "magma", "cividis",
        "coolwarm", "seismic", "RdBu", "hot", "cool", "spring", "summer", "autumn", "winter",
        "gray", "bone", "jet", "nipy_spectral", "set1", "set2", "set3",
    ]
    
    def __init__(self, panel=None):
        super().__init__(panel)
        
        # Color map selection
        self.create_param(
            "colormap",
            "viridis",  # default
            str,
            items=self.COLORMAPS,
            group="Color Map",
        )
        
        # Min/Max values
        self.create_param("vmin", None, (float, type(None)), group="Range")
        self.create_param("vmax", None, (float, type(None)), group="Range")
        
        # Auto-range toggle
        self.create_param("auto_range", True, bool, group="Range")
        
        # Display options
        self.create_param("reverse", False, bool, group="Display")
        self.create_param("climber", True, bool, group="Display")  # Color bar visibility
        self.create_param("show_values", False, bool, group="Display")
    
    def compute(self):
        """Compute updated color map settings."""
        settings = {
            "colormap": self.get_param("colormap"),
            "vmin": self.get_param("vmin"),
            "vmax": self.get_param("vmax"),
            "auto_range": self.get_param("auto_range"),
            "reverse": self.get_param("reverse"),
            "climber": self.get_param("climber"),
            "show_values": self.get_param("show_values"),
        }
        print(f"Color map settings: {settings}")
        return settings
    
    def process_result(self, result):
        """Process the color map settings."""
        self._settings = result
    
    def display_result(self, result):
        """Apply color map settings to the panel."""
        if not self.panel:
            return
        
        # Apply colormap
        if hasattr(self.panel, 'set_colormap'):
            self.panel.set_colormap(result["colormap"])
        
        # Apply range
        if result["auto_range"]:
            if hasattr(self.panel, 'auto_range'):
                self.panel.auto_range()
        else:
            vmin = result["vmin"]
            vmax = result["vmax"]
            if vmin is not None and vmax is not None:
                if hasattr(self.panel, 'set_range'):
                    self.panel.set_range(vmin, vmax)
        
        # Apply reverse
        if result["reverse"]:
            if hasattr(self.panel, 'reverse_colormap'):
                self.panel.reverse_colormap()
        
        print(f"Color map applied: {result['colormap']}")

# Usage:
# from scivianna.panel.panel_2d import Panel2D
# panel = Panel2D(slave=my_slave, name="Colored View")
# ext = ColorMapEditor(panel=panel)
# panel.add_extension(ext)

## Step 6: Complete Example - Data Filter

An extension that filters data based on a threshold value.

In [ ]:
"""
Complete Data Filter Extension

This extension filters polygon data based on a threshold value,
hiding cells below the threshold.
"""

import numpy as np
from scivianna.extension.extension import Extension


class DataFilterExtension(Extension):
    """Extension for filtering data by value threshold."""
    
    def __init__(self, panel=None):
        super().__init__(panel)
        
        # Filter parameters
        self.create_param("threshold", 0.5, float, min=0.0, max=1.0, step=0.01, group="Filter")
        self.create_param("mode", "above", str, items=["above", "below", "range"], group="Filter")
        self.create_param("lower", 0.0, float, group="Filter")
        self.create_param("upper", 1.0, float, group="Filter")
        
        # Display
        self.create_param("enabled", True, bool, group="Display")
    
    def compute(self):
        """Compute filter settings."""
        settings = {
            "threshold": self.get_param("threshold"),
            "mode": self.get_param("mode"),
            "lower": self.get_param("lower"),
            "upper": self.get_param("upper"),
            "enabled": self.get_param("enabled"),
        }
        print(f"Filter settings: {settings}")
        return settings
    
    def process_result(self, result):
        """Process filter results."""
        self._filter_settings = result
    
    def display_result(self, result):
        """Apply filter to the panel."""
        if not self.panel or not result["enabled"]:
            return
        
        # Access slave data
        if not hasattr(self.panel, 'slave'):
            return
        
        slave = self.panel.slave
        interface = slave.get_interface()
        
        # Apply filter to data
        threshold = result["threshold"]
        mode = result["mode"]
        
        if hasattr(interface, 'values'):
            for label, values in interface.values.items():
                if mode == "above":
                    mask = values >= threshold
                elif mode == "below":
                    mask = values < threshold
                elif mode == "range":
                    mask = (values >= result["lower"]) & (values <= result["upper"])
                
                # Update values with filtered data
                interface.values[label] = values[mask]
        
        # Trigger panel update
        if hasattr(self.panel, 'update'):
            self.panel.update()
        
        print(f"Filter applied: mode={mode}, threshold={threshold}")

# Usage:
# ext = DataFilterExtension(panel=my_panel)
# panel.add_extension(ext)

## Step 7: Custom Widget Extension

Create an extension with a completely custom Qt widget.

In [ ]:
"""
Custom Widget Extension Example

This extension adds a custom widget with a button and status display.
"""

from qtpy import QtWidgets
from scivianna.extension.extension import Extension


class CustomStatusWidget(QtWidgets.QWidget):
    """Custom widget for displaying extension status."""
    
    def __init__(self, parent=None):
        super().__init__(parent)
        self.layout = QtWidgets.QVBoxLayout()
        
        # Status label
        self.status_label = QtWidgets.QLabel("Ready")
        self.layout.addWidget(self.status_label)
        
        # Action button
        self.action_button = QtWidgets.QPushButton("Run Analysis")
        self.layout.addWidget(self.action_button)
        
        # Connect button to action
        self.action_button.clicked.connect(self._on_action)
        
        self.setLayout(self.layout)
    
    def _on_action(self):
        """Handle button click."""
        self.status_label.setText("Analyzing...")
        # Trigger extension compute
        if hasattr(self, 'extension'):
            self.extension.compute()
        self.status_label.setText("Analysis Complete")
    
    def set_status(self, status: str):
        """Update status display."""
        self.status_label.setText(status)


class CustomWidgetExtension(Extension):
    """Extension with a custom Qt widget."""
    
    def __init__(self, panel=None):
        super().__init__(panel)
        
        # Regular parameters
        self.create_param("analysis_type", "quick", str, items=["quick", "thorough"], group="Analysis")
        
        # Register custom widget
        self.register_widget(
            "custom_status",
            CustomStatusWidget,
            group="Status",
        )
        
        # Link custom widget to extension
        self.custom_widget = None
    
    def _create_widget(self, widget_class):
        """Override to link custom widget."""
        widget = widget_class()
        if isinstance(widget, CustomStatusWidget):
            widget.extension = self
            self.custom_widget = widget
        return widget
    
    def compute(self):
        """Run analysis."""
        analysis_type = self.get_param("analysis_type")
        result = {
            "analysis_type": analysis_type,
            "status": "completed",
        }
        
        # Update custom widget
        if self.custom_widget:
            self.custom_widget.set_status(f"Analysis: {analysis_type}")
        
        return result

# Usage:
# ext = CustomWidgetExtension(panel=my_panel)
# panel.add_extension(ext)

## Step 8: Asynchronous Computation

For long-running computations, use the async computation pattern.

In [ ]:
"""
Asynchronous Computation Extension

This extension demonstrates how to handle long-running computations
without blocking the UI.
"""

import time
import threading
from scivianna.extension.extension import Extension


class AsyncComputationExtension(Extension):
    """Extension with asynchronous computation."""
    
    def __init__(self, panel=None):
        super().__init__(panel)
        
        self.create_param("compute_time", 2.0, float, min=0.1, max=10.0, step=0.1, group="Computation")
        self.create_param("iterations", 5, int, min=1, max=20, group="Computation")
        self.create_param("running", False, bool, readonly=True, group="Status")
    
    def compute(self):
        """Start asynchronous computation."""
        compute_time = self.get_param("compute_time")
        iterations = self.get_param("iterations")
        
        # Mark as running
        self.set_param("running", True)
        
        # Run computation in background thread
        def run_computation():
            results = []
            for i in range(iterations):
                time.sleep(compute_time / iterations)
                progress = (i + 1) / iterations * 100
                results.append({
                    "iteration": i + 1,
                    "progress": progress,
                    "value": i * compute_time,
                })
            
            # Call process_result on the main thread
            import qtpy.QtCore as qc
            qc.QMetaObject.invokeMethod(
                self,
                "_process_results",
                qc.Qt.ConnectionType.QueuedConnection,
                qc.Q_ARG(list, results),
            )
        
        thread = threading.Thread(target=run_computation, daemon=True)
        thread.start()
        return {"status": "running", "iterations": iterations}
    
    def _process_results(self, results):
        """Process results on the main thread."""
        self.set_param("running", False)
        print(f"Computation complete: {len(results)} iterations")
        self.process_result(results)
    
    def process_result(self, result):
        """Handle computation result."""
        self._last_results = result
    
    def display_result(self, result):
        """Update panel with results."""
        if self.panel and hasattr(self.panel, 'update'):
            self.panel.update()

# Usage:
# ext = AsyncComputationExtension(panel=my_panel)
# panel.add_extension(ext)

## Extension Development Checklist

When creating a custom extension, ensure:

- [ ] **Inherit from `Extension`**: Base class for all extensions
- [ ] **Call `super().__init__(panel)`**: Initialize parent class
- [ ] **Create parameters with `create_param`**: Define UI widgets
- [ ] **Implement `compute`**: Return result dict when params change
- [ ] **Implement `process_result`**: Handle computation results
- [ ] **Implement `display_result`**: Update panel visualization
- [ ] **Use parameter grouping**: Organize related parameters
- [ ] **Access slave data**: Via `self.panel.slave`
- [ ] **Handle async computations**: Use threading for long tasks
- [ ] **Register custom widgets**: With `register_widget` if needed

## Parameter Types Reference

| Type | UI Widget |
|------|-----------|
| `str` | LineEdit (or ComboBox with `items`) |
| `bool` | CheckBox |
| `int` | SpinBox |
| `float` | DoubleSpinBox |
| `tuple` (RGB) | QColorButton |
| `Path` | File/Dir chooser |

## Extension Methods Reference

| Method | Purpose |
|--------|---------|
| `create_param(name, default, type, ...)` | Create a new parameter |
| `get_param(name)` | Get current parameter value |
| `set_param(name, value)` | Set parameter value (from code) |
| `compute()` | Called on parameter change |
| `process_result(result)` | Handle computation result |
| `display_result(result)` | Update panel visualization |
| `register_widget(name, widget_class, ...)` | Register custom Qt widget |

## Summary

In this tutorial, you learned:
1. The extension architecture in scivianna
2. How to create simple extensions with UI widgets
3. Different parameter types and their widgets
4. How to group parameters for better organization
5. How to access slave data from extensions
6. Complete examples: Color Map Editor and Data Filter
7. How to create custom Qt widgets
8. How to handle asynchronous computations

## Next Steps

- [Tutorial 6: PyQt Fluent Widgets (User)](./6_matplotlib_api.ipynb)